# Accounting Transaction Failure Review
### Student project notebook — no Python programming required

Use the feature CSV exported from your Excel workbook. Change only the clearly marked settings, run cells from top to bottom, and interpret the outputs.

The notebook compares logistic regression and a decision tree. It selects a probability threshold using validation data and explicit financial assumptions. Final test data is kept separate and evaluated only after you lock your choices.

**Important:** records are simulated. If results show no useful signal, report that finding; do not alter outcomes or tune against the final test set.

## 1. Before you start

Place these files in the same folder as this notebook:

- features_development.csv: your Excel-prepared development data (normally 80% of all records).
- features_final_test.csv: a separate instructor-held 20% test file, supplied only for final evaluation.

The CSV must have one row per transaction and retain Transaction Outcome with values 1 = success and 0 = failure. Keep Transaction ID as a record key if supplied. Export the same Excel-created feature columns for final test data.

The class environment needs pandas, scikit-learn and matplotlib. You do not need to write or edit Python code. If a cell reports an error, check the file name, column names and CSV export settings first.

## 2. Student settings — edit only this cell

Choose only features known before processing. Do not include the target, transaction ID, processing time, accuracy score, or the supplied normalized amount.

Keep the main training share at 70%. Compare 60% and 80% only as limited sensitivity analyses by changing TRAIN_SHARE and rerunning from this cell. Do not use the final test file for any choice.

To compare your Excel feature engineering, first run with the two original eligible columns only and save the validation table. Then add your engineered column names to FEATURE_COLUMNS, rerun from the settings cell, and compare the same scenario/model/threshold results. Record both runs in your experiment log.


In [ ]:
# === STUDENT SETTINGS ===
DATA_FILE = "features_development.csv"
TARGET_COLUMN = "Transaction Outcome"
ID_COLUMNS = ["Transaction ID"]

# Select approved original and Excel-engineered columns.
FEATURE_COLUMNS = [
    "Account Type",
    "Transaction Amount",
    # Add exact names of your Excel-engineered features here.
]

TRAIN_SHARE = 0.70       # choose 0.60, 0.70, or 0.80 of development data
SPLIT_SEED = 42          # keep fixed for fair comparisons
MODELS_TO_COMPARE = ["logistic_regression", "decision_tree"]
TREE_DEPTHS = [2, 4, 6]
THRESHOLDS = [round(x / 100, 2) for x in range(5, 100, 5)]

## 3. Load and check the development data

The target is converted to Failure: 1 means failure and 0 means success. The ID is not used as a predictor. Potentially post-processing columns are blocked automatically. Missing values and categorical fields are handled inside the model pipeline, with preprocessing fitted on training data only.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

if TRAIN_SHARE not in (0.60, 0.70, 0.80):
    raise ValueError("TRAIN_SHARE must be 0.60, 0.70, or 0.80.")
if not os.path.exists(DATA_FILE):
    raise FileNotFoundError(f"Could not find {DATA_FILE!r}. Place it beside this notebook.")

data = pd.read_csv(DATA_FILE)
print(f"Rows: {len(data):,} | Columns: {len(data.columns)}")
display(data.head())

In [ ]:
if TARGET_COLUMN not in data.columns:
    raise ValueError(f"Target column {TARGET_COLUMN!r} was not found.")
if data[TARGET_COLUMN].isna().any():
    raise ValueError("The target contains missing values. Ask the instructor how to handle them.")
if not set(data[TARGET_COLUMN].dropna().unique()).issubset({0, 1}):
    raise ValueError("Transaction Outcome must contain only 0 (failure) and 1 (success).")

data = data.copy()
data["Failure"] = 1 - data[TARGET_COLUMN].astype(int)
blocked = {TARGET_COLUMN, "Failure", "Processing Time (seconds)",
           "Accuracy Score", "Normalized Transaction Amount"}
blocked.update(ID_COLUMNS)
selected = list(dict.fromkeys(FEATURE_COLUMNS))
if not selected or selected == ["Account Type", "Transaction Amount"]:
    raise ValueError("Edit FEATURE_COLUMNS to include the features you intend to evaluate.")
missing_cols = [c for c in selected if c not in data.columns]
if missing_cols:
    raise ValueError(f"Selected columns not found: {missing_cols}")
prohibited = [c for c in selected if c in blocked]
if prohibited:
    raise ValueError(f"Remove prohibited target, ID, or post-processing columns: {prohibited}")

X = data[selected].copy()
y = data["Failure"].copy()
class_counts = y.value_counts().reindex([0, 1], fill_value=0)
display(pd.DataFrame({
    "Class": ["Success (0)", "Failure (1)"],
    "Count": [int(class_counts[0]), int(class_counts[1])],
    "Share": [class_counts[0] / len(y), class_counts[1] / len(y)]
}))
print("Missing values among selected predictors:")
display(X.isna().sum().rename("Missing count").to_frame())
if y.nunique() < 2:
    raise ValueError("Only one target class is present; classification is not possible.")

## 4. Create training and validation sets

The development file should exclude the final test data. A fixed seed and stratification make the split reproducible while preserving the failure share as far as possible. The notebook never uses the final test file in these comparisons.

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, train_size=TRAIN_SHARE, random_state=SPLIT_SEED, stratify=y
)
print(f"Training rows: {len(X_train)} | Validation rows: {len(X_valid)}")
print(f"Failure share — train: {y_train.mean():.3f} | validation: {y_valid.mean():.3f}")
if y_train.nunique() < 2 or y_valid.nunique() < 2:
    raise ValueError("Both classes are needed in training and validation. Ask the instructor.")

## 5. Build and compare models

The same training and validation rows are used for each model. Numeric fields are median-imputed and standardised; categorical fields are mode-imputed and one-hot encoded. Transformations are learned on training rows only. Logistic regression is the simpler linear model; the decision tree can represent non-linear splits and is tested at the specified depths.

In [ ]:
numeric_columns = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_columns = [c for c in X_train.columns if c not in numeric_columns]

def make_preprocessor():
    transformers = []
    if numeric_columns:
        numeric_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ])
        transformers.append(("numeric", numeric_pipe, numeric_columns))
    if categorical_columns:
        categorical_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ])
        transformers.append(("categorical", categorical_pipe, categorical_columns))
    return ColumnTransformer(transformers=transformers, remainder="drop")

candidate_models = {}
if "logistic_regression" in MODELS_TO_COMPARE:
    candidate_models["Logistic regression"] = Pipeline([
        ("preprocess", make_preprocessor()),
        ("model", LogisticRegression(max_iter=2000, random_state=SPLIT_SEED))
    ])
if "decision_tree" in MODELS_TO_COMPARE:
    for depth in TREE_DEPTHS:
        candidate_models[f"Decision tree (depth {depth})"] = Pipeline([
            ("preprocess", make_preprocessor()),
            ("model", DecisionTreeClassifier(max_depth=depth, random_state=SPLIT_SEED))
        ])
if not candidate_models:
    raise ValueError("Choose at least one model in MODELS_TO_COMPARE.")

validation_probabilities = {}
for name, pipeline in candidate_models.items():
    pipeline.fit(X_train, y_train)
    positive_class_index = list(pipeline.named_steps["model"].classes_).index(1)
    validation_probabilities[name] = pipeline.predict_proba(X_valid)[:, positive_class_index]
print(f"Models fitted: {len(candidate_models)}")

## 6. Compare thresholds under the three cost scenarios

A transaction is flagged when its estimated failure risk is at least the threshold. Lower thresholds usually flag more transactions and may catch more failures, while also creating more false alerts.

The costs are fictional teaching assumptions. This simplified score counts false negatives and false positives only; it does not include the cost of reviewing true positives.

In [ ]:
COST_SCENARIOS = {
    "A — Symmetric error costs": {"FN": 20, "FP": 20},
    "B — Expensive missed failures": {"FN": 200, "FP": 10},
    "C — Expensive unnecessary alerts": {"FN": 50, "FP": 40},
}

def get_metrics(y_true, probabilities, threshold, fn_cost, fp_cost):
    predictions = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predictions, labels=[0, 1]).ravel()
    precision = tp / (tp + fp) if tp + fp else np.nan
    recall = tp / (tp + fn) if tp + fn else np.nan
    f1 = (2 * tp / (2 * tp + fp + fn)) if (2 * tp + fp + fn) else np.nan
    cost = fn_cost * fn + fp_cost * fp
    n = len(y_true)
    return {
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "Accuracy": (tp + tn) / n, "Precision": precision, "Recall": recall, "F1": f1,
        "Review rate": (tp + fp) / n,
        "Cost": cost, "Cost per 1,000": 1000 * cost / n
    }

rows = []
for scenario, costs in COST_SCENARIOS.items():
    for model_name, probs in validation_probabilities.items():
        for threshold in THRESHOLDS:
            result = get_metrics(y_valid.to_numpy(), probs, threshold, costs["FN"], costs["FP"])
            rows.append({"Scenario": scenario, "Model": model_name,
                         "Threshold": threshold, **result})
validation_results = pd.DataFrame(rows)

In [ ]:
n_failure = int((y_valid == 1).sum())
n_success = int((y_valid == 0).sum())
baseline_rows = []
for scenario, costs in COST_SCENARIOS.items():
    baseline_rows.extend([
        {"Scenario": scenario, "Policy": "Flag none", "Cost": costs["FN"] * n_failure,
         "Cost per 1,000": 1000 * costs["FN"] * n_failure / len(y_valid), "Review rate": 0.0},
        {"Scenario": scenario, "Policy": "Flag all", "Cost": costs["FP"] * n_success,
         "Cost per 1,000": 1000 * costs["FP"] * n_success / len(y_valid), "Review rate": 1.0},
    ])
baselines = pd.DataFrame(baseline_rows)
best_rows = []
for scenario in COST_SCENARIOS:
    part = validation_results[validation_results["Scenario"] == scenario]
    best = part.sort_values(["Cost", "Review rate", "Threshold"],
                            ascending=[True, True, False]).iloc[0]
    baseline_best = baselines[baselines["Scenario"] == scenario]["Cost"].min()
    best_rows.append({
        "Scenario": scenario, "Selected model": best["Model"],
        "Threshold": best["Threshold"], "TP": int(best["TP"]), "FP": int(best["FP"]),
        "FN": int(best["FN"]), "TN": int(best["TN"]),
        "Accuracy": best["Accuracy"], "Precision": best["Precision"], "Recall": best["Recall"], "F1": best["F1"],
        "Review rate": best["Review rate"], "Cost per 1,000": best["Cost per 1,000"],
        "Best simple baseline cost / 1,000": 1000 * baseline_best / len(y_valid)
    })
validation_choices = pd.DataFrame(best_rows)
display(validation_choices.round(3))

### Interpret the validation results

1. Does the preferred model or threshold change across cost scenarios?
2. Does threshold choice change cost more than model choice?
3. Does the selected policy beat both simple baselines?
4. How much review capacity would it use?
5. Do your Excel-engineered features improve the cost measure?

Automatic selection is a starting point. If you choose a different policy for a documented business reason, record that choice and validation cost. Do not choose a model based on final-test results.

In [ ]:
fig, axes = plt.subplots(1, len(COST_SCENARIOS), figsize=(16, 4))
for ax, (scenario, costs) in zip(axes, COST_SCENARIOS.items()):
    for model_name, probs in validation_probabilities.items():
        curve = []
        for threshold in THRESHOLDS:
            m = get_metrics(y_valid.to_numpy(), probs, threshold, costs["FN"], costs["FP"])
            curve.append((threshold, m["Cost per 1,000"]))
        curve = np.array(curve)
        ax.plot(curve[:, 0], curve[:, 1], marker="o", markersize=3, label=model_name)
    ax.set_title(scenario)
    ax.set_xlabel("Decision threshold")
    ax.set_ylabel("Error cost per 1,000")
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 7. Optional sensitivity analysis

To investigate training size, change TRAIN_SHARE to another allowed value in the settings cell and rerun from there. Keep the seed fixed. Compare validation cost and policy cautiously: a new split also changes which observations appear in validation.

Use 70% as your main result. Do not keep trying split settings until one produces a favourable outcome.

## 8. Lock choices, then evaluate the final test set once

Complete and save your validation analysis first. Agree on the selected model and threshold for each scenario, then enter those choices below and set RUN_FINAL_TEST to True. This cell loads the separate instructor test file and reports results for those locked choices only.

Do not rerun selection based on test results. If your instructor has not provided a final test file, leave this step unrun and submit your validation analysis.

In [ ]:
# === FINAL TEST: edit only after locking your validation choices ===
RUN_FINAL_TEST = False
FINAL_TEST_FILE = "features_final_test.csv"

LOCKED_SELECTIONS = {
    "A — Symmetric error costs": {"model": "Logistic regression", "threshold": 0.50},
    "B — Expensive missed failures": {"model": "Logistic regression", "threshold": 0.50},
    "C — Expensive unnecessary alerts": {"model": "Logistic regression", "threshold": 0.50},
}

if RUN_FINAL_TEST:
    if not os.path.exists(FINAL_TEST_FILE):
        raise FileNotFoundError("Final test file not found. Ask your instructor.")
    test_data = pd.read_csv(FINAL_TEST_FILE)
    if TARGET_COLUMN not in test_data.columns:
        raise ValueError(f"Final test file must contain {TARGET_COLUMN!r}.")
    if test_data[TARGET_COLUMN].isna().any():
        raise ValueError("Final test target has missing values.")
    if not set(test_data[TARGET_COLUMN].unique()).issubset({0, 1}):
        raise ValueError("Final test outcome must use 0 (failure) and 1 (success).")
    absent = [c for c in selected if c not in test_data.columns]
    if absent:
        raise ValueError(f"Final test file is missing selected columns: {absent}")
    X_test = test_data[selected].copy()
    y_test = 1 - test_data[TARGET_COLUMN].astype(int)
    test_rows = []
    for scenario, costs in COST_SCENARIOS.items():
        choice = LOCKED_SELECTIONS[scenario]
        model_name, threshold = choice["model"], float(choice["threshold"])
        if model_name not in candidate_models:
            raise ValueError(f"{model_name!r} was not fitted. Check model settings.")
        if not 0 < threshold < 1:
            raise ValueError("Each locked threshold must be strictly between 0 and 1.")
        pipeline = candidate_models[model_name]
        class_index = list(pipeline.named_steps["model"].classes_).index(1)
        probabilities = pipeline.predict_proba(X_test)[:, class_index]
        m = get_metrics(y_test.to_numpy(), probabilities, threshold, costs["FN"], costs["FP"])
        test_rows.append({"Scenario": scenario, "Model": model_name,
                          "Threshold": threshold, **m})
    final_test_results = pd.DataFrame(test_rows)
    display(final_test_results.round(3))
    print("Interpret this once as final evaluation. Do not retune from these results.")
else:
    print("Final test not run. Lock your model and threshold choices first.")

## 9. Export a compact experiment log

Record configurations you actually tried. Add Excel feature-set comparisons and any sensitivity runs before submitting. The log should make choices traceable.

In [ ]:
experiment_log = validation_results[
    ["Scenario", "Model", "Threshold", "TP", "FP", "FN", "TN",
     "Accuracy", "Precision", "Recall", "F1", "Review rate", "Cost per 1,000"]
].copy()
display(experiment_log.head(15).round(3))
# Optional: uncomment to save the full threshold comparison table.
# experiment_log.to_csv("validation_experiment_log.csv", index=False)

## Submission checklist

- [ ] Excel workbook includes unchanged raw data, feature sheet and feature dictionary.
- [ ] CSV column names match FEATURE_COLUMNS.
- [ ] Main results use 70% of development data for training.
- [ ] Logistic regression and decision trees were compared on the same split.
- [ ] All three cost scenarios and both simple baselines are discussed.
- [ ] Threshold choices were made using validation data only.
- [ ] Final test results, if supplied, were run only after locking choices.
- [ ] The management memo explains assumptions, review workload and limitations.